In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!pip install torch torchvision numpy opencv-python matplotlib tqdm


In [3]:
import sys, os

# Remove conflicting utils.py
if os.path.exists("/content/utils.py"):
    os.remove("/content/utils.py")

TRAINING_DIR = "/content/drive/MyDrive/project_Landset08/training"
sys.path.append(TRAINING_DIR)

print("✅ Training directory added to path")


✅ Training directory added to path


STEP 5: VERIFY FINAL DATASET (NO TRAINING YET)

In [4]:
DATASET_DIR = "/content/drive/MyDrive/project_Landset08/final_dataset"

print(os.listdir(DATASET_DIR))
print(os.listdir(os.path.join(DATASET_DIR, "splits")))


['HR_Thermal', 'LR_Thermal', 'HR_Optical', 'splits']
['train.txt', 'val.txt', 'test.txt']


STEP 6: WRITE dataset.py (FINAL – DO NOT CHANGE)

In [9]:
%%writefile /content/drive/MyDrive/project_Landset08/training/dataset.py
import os
import torch
import numpy as np
from torch.utils.data import Dataset

class ThermalDataset(Dataset):
    def __init__(self, base_dir, split_file):
        self.lr_dir = os.path.join(base_dir, "LR_Thermal")
        self.rgb_dir = os.path.join(base_dir, "HR_Optical")
        self.hr_dir = os.path.join(base_dir, "HR_Thermal")

        with open(split_file, 'r') as f:
            self.ids = [line.strip() for line in f]

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]

        # Modified to include suffixes for correct filename matching
        lr = np.load(os.path.join(self.lr_dir, img_id + "_LR.npy"))
        rgb = np.load(os.path.join(self.rgb_dir, img_id + "_RGB.npy"))
        hr = np.load(os.path.join(self.hr_dir, img_id + "_HR.npy"))

        lr = torch.FloatTensor(lr).unsqueeze(0) / 255.0
        hr = torch.FloatTensor(hr).unsqueeze(0) / 255.0
        rgb = torch.FloatTensor(rgb).permute(2, 0, 1) / 255.0

        return lr, rgb, hr


Overwriting /content/drive/MyDrive/project_Landset08/training/dataset.py


STEP 7: TEST DATASET (MANDATORY)

In [13]:
import os
import importlib
# Import the dataset module itself, not just ThermalDataset from it
import dataset

# Reload the dataset module to ensure the latest changes are used
importlib.reload(dataset)

# Access ThermalDataset from the reloaded module
from dataset import ThermalDataset

base_dir = "/content/drive/MyDrive/project_Landset08/final_dataset"
split_file = base_dir + "/splits/train.txt"

ds = ThermalDataset(base_dir, split_file)
print(f"Dataset length: {len(ds)}")

# --- Diagnostic Code Start ---
if len(ds) > 0:
    # Accessing the internal ids list from the dataset instance
    first_img_id = ds.ids[0]
    # The expected path should now include the _LR suffix for LR_Thermal
    expected_lr_path = os.path.join(ds.lr_dir, first_img_id + "_LR.npy")
    print(f"Attempting to load (corrected suffix): {expected_lr_path}")
    if not os.path.exists(expected_lr_path):
        print(f"Error: File not found at {expected_lr_path}")
        print(f"--- Contents of LR_Thermal directory ({ds.lr_dir}): ---")
        try:
            lr_files = [f for f in os.listdir(ds.lr_dir) if f.endswith('.npy')]
            print(f"Found {len(lr_files)} .npy files. First 5: {lr_files[:5]}")
            if first_img_id + "_LR.npy" not in lr_files:
                print(f"'{first_img_id}_LR.npy' is NOT in the LR_Thermal directory.")
        except FileNotFoundError:
            print(f"Directory {ds.lr_dir} itself was not found.")
        except Exception as e:
            print(f"Could not list directory contents: {e}")
        print(f"--- First 5 IDs from train.txt: {ds.ids[:5]} ---")
        print("Please verify that the file '0000569_LR.npy' exists in the 'LR_Thermal' folder")
        print("and that '0000569' is indeed a valid ID in 'train.txt'.")
    else:
        print(f"File {expected_lr_path} exists. Proceeding with loading.")
# --- Diagnostic Code End ---

lr, rgb, hr = ds[0]
print(lr.shape, rgb.shape, hr.shape)


Dataset length: 462
Attempting to load (corrected suffix): /content/drive/MyDrive/project_Landset08/final_dataset/LR_Thermal/0000569_LR.npy
File /content/drive/MyDrive/project_Landset08/final_dataset/LR_Thermal/0000569_LR.npy exists. Proceeding with loading.
torch.Size([1, 264, 264]) torch.Size([3, 264, 264]) torch.Size([1, 264, 264])


test dataloader

In [14]:
from torch.utils.data import DataLoader

loader = DataLoader(ds, batch_size=4, shuffle=True)

lr, rgb, hr = next(iter(loader))
print(lr.shape, rgb.shape, hr.shape)


torch.Size([4, 1, 264, 264]) torch.Size([4, 3, 264, 264]) torch.Size([4, 1, 264, 264])


model.py

In [15]:
%%writefile /content/drive/MyDrive/project_Landset08/training/model.py
import torch
import torch.nn as nn
import torch.nn.functional as F


class ResidualBlock(nn.Module):
    """Standard residual block used for feature refinement."""
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.act = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)

    def forward(self, x):
        identity = x
        out = self.conv1(x)
        out = self.act(out)
        out = self.conv2(out)
        return identity + out


class ChannelAttention(nn.Module):
    """Squeeze-and-Excitation style channel attention."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        w = self.pool(x).view(b, c)
        w = self.fc(w).view(b, c, 1, 1)
        return x * w


class FusionBlock(nn.Module):
    """Fuse thermal and optical features."""
    def __init__(self, in_ch_thermal, in_ch_optical, feat_channels=64, n_resblocks=4):
        super().__init__()
        self.t_head = nn.Conv2d(in_ch_thermal, feat_channels, 3, padding=1)
        self.o_head = nn.Conv2d(in_ch_optical, feat_channels, 3, padding=1)
        blocks = []
        for _ in range(n_resblocks):
            blocks.append(ResidualBlock(feat_channels * 2))
        self.fusion_res = nn.Sequential(*blocks)
        self.reduce = nn.Conv2d(feat_channels * 2, feat_channels, 1)
        self.ca = ChannelAttention(feat_channels)

    def forward(self, t, o):
        tf = self.t_head(t)
        of = self.o_head(o)
        fused = torch.cat([tf, of], dim=1)
        fused = self.fusion_res(fused)
        fused = self.reduce(fused)
        fused = self.ca(fused)
        return fused


class Upsampler(nn.Module):
    """Flexible upsample block."""
    def __init__(self, in_channels, out_channels=1, up_scale=1):
        super().__init__()
        self.up_scale = up_scale
        if up_scale == 1:
            self.net = nn.Sequential(
                ResidualBlock(in_channels),
                nn.Conv2d(in_channels, out_channels, 3, padding=1),
            )
        else:
            mid_ch = in_channels * (up_scale ** 2)
            self.net = nn.Sequential(
                nn.Conv2d(in_channels, mid_ch, 3, padding=1),
                nn.PixelShuffle(up_scale),
                nn.ReLU(inplace=True),
                nn.Conv2d(in_channels, out_channels, 3, padding=1),
            )

    def forward(self, x):
        return self.net(x)


class ThermalFusionNet(nn.Module):
    """Optical-guided thermal super-resolution network."""
    def __init__(self, feat_channels=64, n_refine=6, up_scale=1):
        super().__init__()
        self.fusion = FusionBlock(1, 3, feat_channels)
        refine_blocks = []
        for _ in range(n_refine):
            refine_blocks.append(ResidualBlock(feat_channels))
        self.refine = nn.Sequential(*refine_blocks)
        self.upsampler = Upsampler(feat_channels, 1, up_scale)
        self.final_conv = nn.Conv2d(1, 1, 3, padding=1)

    def forward(self, lr_thermal, hr_optical):
        fused = self.fusion(lr_thermal, hr_optical)
        refined = self.refine(fused)
        sr = self.upsampler(refined)
        if sr.shape == lr_thermal.shape:
            out = sr + self.final_conv(lr_thermal)
        else:
            up_lr = F.interpolate(lr_thermal, size=sr.shape[2:], mode='bilinear', align_corners=False)
            out = sr + self.final_conv(up_lr)
        return out


if __name__ == "__main__":
    # Quick smoke test
    model = ThermalFusionNet(feat_channels=64, n_refine=4, up_scale=1)
    lr = torch.randn(2, 1, 264, 264)
    rgb = torch.randn(2, 3, 264, 264)
    out = model(lr, rgb)
    print("out", out.shape)  # expected (2,1,264,264)


Writing /content/drive/MyDrive/project_Landset08/training/model.py


In [16]:
from model import ThermalFusionNet

net = ThermalFusionNet()


train.py

In [17]:
%%writefile /content/drive/MyDrive/project_Landset08/training/train.py
import os
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
from dataset import ThermalDataset
from model import ThermalFusionNet

# -------------------------
# Config / Hyperparameters
# -------------------------
BASE_DIR = "/content/drive/MyDrive/project_Landset08/final_dataset"
SPLIT_FILE = os.path.join(BASE_DIR, "splits/train.txt")
BATCH_SIZE = 4
NUM_EPOCHS = 50
LEARNING_RATE = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT_DIR = "/content/drive/MyDrive/project_Landset08/training/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# -------------------------
# Dataset & DataLoader
# -------------------------
dataset = ThermalDataset(BASE_DIR, SPLIT_FILE)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

print(f"Dataset length: {len(dataset)}")
print(f"Using device: {DEVICE}")

# -------------------------
# Model, Loss, Optimizer
# -------------------------
model = ThermalFusionNet(feat_channels=64, n_refine=4, up_scale=1).to(DEVICE)
criterion = nn.L1Loss()  # regression for thermal values
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# -------------------------
# Training Loop
# -------------------------
for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for i, (lr, rgb, hr) in enumerate(loader, 1):
        lr = lr.to(DEVICE)
        rgb = rgb.to(DEVICE)
        hr = hr.to(DEVICE)

        optimizer.zero_grad()
        sr = model(lr, rgb)
        loss = criterion(sr, hr)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if i % 20 == 0:
            print(f"[Epoch {epoch}/{NUM_EPOCHS}] Step {i}/{len(loader)} - Loss: {running_loss / i:.6f}")

    # Save checkpoint at end of each epoch
    ckpt_path = os.path.join(CHECKPOINT_DIR, f"epoch_{epoch:03d}.pt")
    torch.save(model.state_dict(), ckpt_path)
    print(f"Saved checkpoint: {ckpt_path} - Avg Loss: {running_loss / len(loader):.6f}")

print("Training finished.")


Writing /content/drive/MyDrive/project_Landset08/training/train.py


now train

In [18]:
!python /content/drive/MyDrive/project_Landset08/training/train.py


Dataset length: 462
Using device: cuda
[Epoch 1/50] Step 20/116 - Loss: 0.155488
[Epoch 1/50] Step 40/116 - Loss: 0.093297
[Epoch 1/50] Step 60/116 - Loss: 0.067128
[Epoch 1/50] Step 80/116 - Loss: 0.053867
[Epoch 1/50] Step 100/116 - Loss: 0.046836
Saved checkpoint: /content/drive/MyDrive/project_Landset08/training/checkpoints/epoch_001.pt - Avg Loss: 0.042278
[Epoch 2/50] Step 20/116 - Loss: 0.012372
[Epoch 2/50] Step 40/116 - Loss: 0.011024
[Epoch 2/50] Step 60/116 - Loss: 0.011446
[Epoch 2/50] Step 80/116 - Loss: 0.010708
[Epoch 2/50] Step 100/116 - Loss: 0.010882
Saved checkpoint: /content/drive/MyDrive/project_Landset08/training/checkpoints/epoch_002.pt - Avg Loss: 0.012658
[Epoch 3/50] Step 20/116 - Loss: 0.014145
[Epoch 3/50] Step 40/116 - Loss: 0.012690
[Epoch 3/50] Step 60/116 - Loss: 0.011511
[Epoch 3/50] Step 80/116 - Loss: 0.009819
[Epoch 3/50] Step 100/116 - Loss: 0.008681
Saved checkpoint: /content/drive/MyDrive/project_Landset08/training/checkpoints/epoch_003.pt - Avg L

In [19]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.current_device())
print(torch.cuda.get_device_name(torch.cuda.current_device()))


True
0
Tesla T4


In [21]:
import os

ckpt_dir = "/content/drive/MyDrive/project_Landset08/training/checkpoints"
os.listdir(ckpt_dir)


['epoch_001.pt',
 'epoch_002.pt',
 'epoch_003.pt',
 'epoch_004.pt',
 'epoch_005.pt',
 'epoch_006.pt',
 'epoch_007.pt',
 'epoch_008.pt',
 'epoch_009.pt',
 'epoch_010.pt',
 'epoch_011.pt',
 'epoch_012.pt',
 'epoch_013.pt',
 'epoch_014.pt',
 'epoch_015.pt',
 'epoch_016.pt',
 'epoch_017.pt',
 'epoch_018.pt',
 'epoch_019.pt',
 'epoch_020.pt',
 'epoch_021.pt',
 'epoch_022.pt',
 'epoch_023.pt',
 'epoch_024.pt',
 'epoch_025.pt',
 'epoch_026.pt',
 'epoch_027.pt',
 'epoch_028.pt',
 'epoch_029.pt',
 'epoch_030.pt',
 'epoch_031.pt',
 'epoch_032.pt',
 'epoch_033.pt',
 'epoch_034.pt',
 'epoch_035.pt',
 'epoch_036.pt',
 'epoch_037.pt',
 'epoch_038.pt',
 'epoch_039.pt',
 'epoch_040.pt',
 'epoch_041.pt',
 'epoch_042.pt',
 'epoch_043.pt',
 'epoch_044.pt',
 'epoch_045.pt',
 'epoch_046.pt',
 'epoch_047.pt',
 'epoch_048.pt',
 'epoch_049.pt',
 'epoch_050.pt']